# Gan 2026 Living Observatory

Purpose: keep one runnable notebook for loading checks, gold-label distribution, scoring, and failure slices while candidate pipelines evolve.

This is a development-control surface, not a benchmark claim. Use it to inspect train/validation behavior, candidate artifacts, and row families; keep the locked test split out of routine analysis.

Core docs to keep in view:

- `PROJECT_STATUS.md` for the current objective, work board, and claim caveats.
- `docs/research/contribution_thesis.md` for the modular, generalizable, transparent hybrid thesis.
- `docs/design/data_contract.md` for row validity, source labels, and benchmark data handling.
- `docs/design/gan2026_split_protocol.md` for split discipline and validation-escalation rules.
- `docs/research/gan2026_architecture_space_2026-06-01.md` for the architecture gate before the metric gate.

## Setup

Run from either the repo root or the `notebooks/` directory. The cell finds the repo root and uses the package APIs rather than reimplementing loading or scoring logic.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find repo root from current working directory")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from clinical_extraction.tasks.seizure_frequency.gan2026.data import (  # noqa: E402
    DEFAULT_DATA_PATH,
    DEFAULT_SPLIT_MANIFEST_PATH,
    load_records_for_split,
    load_split_manifest,
)
from clinical_extraction.tasks.seizure_frequency.gan2026.evaluate import (  # noqa: E402
    convert_to_categories,
    evaluate_predictions,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

DATA_PATH = REPO_ROOT / DEFAULT_DATA_PATH
SPLIT_MANIFEST_PATH = REPO_ROOT / DEFAULT_SPLIT_MANIFEST_PATH
manifest = load_split_manifest(SPLIT_MANIFEST_PATH)

REPO_ROOT

PosixPath('/Users/cobro/code/clinical-extraction')

## Loading And Split Shape

This pins the basic data contract: source row index, split membership, `row_ok`, gold label semantics, and scorer categories.

In [10]:
def records_to_frame(records, split: str | None = None) -> pd.DataFrame:
    rows = []
    for record in records:
        rows.append(
            {
                "split": split,
                "source_row_index": record.source_row_index,
                "gold_label": record.gold_label,
                "gold_normalized_label": record.gold_normalized_label,
                "gold_label_kind": str(record.gold_label_kind),
                "gold_monthly_frequency": record.gold_monthly_frequency,
                "gold_purist_category": convert_to_categories(
                    [record.gold_monthly_frequency], method="purist"
                )[0],
                "gold_pragmatic_category": convert_to_categories(
                    [record.gold_monthly_frequency], method="pragmatic"
                )[0],
                "note_chars": len(record.note_text),
                "gold_reference_chars": len(record.gold_reference),
            }
        )
    return pd.DataFrame(rows)


split_frames = {
    split: records_to_frame(
        load_records_for_split(split, data_path=DATA_PATH, manifest_path=SPLIT_MANIFEST_PATH),
        split=split,
    )
    for split in manifest["splits"]
}
all_records_df = pd.concat(split_frames.values(), ignore_index=True)

split_summary = (
    all_records_df.groupby("split", observed=True)
    .agg(
        rows=("source_row_index", "count"),
        mean_note_chars=("note_chars", "mean"),
    )
    .assign(mean_note_chars=lambda frame: frame["mean_note_chars"].round(1))
)
split_summary

,rows,mean_note_chars
split,,
test,450,2752.6
train,300,2774.0
validation,750,2735.5


## Gold-Label Distribution

These tables show the target surface before candidate behavior enters the picture. Use them when interpreting score movement: a candidate can look good overall while failing a clinically important sparse slice.

In [3]:
def distribution_table(df: pd.DataFrame, column: str) -> pd.DataFrame:
    counts = df.groupby(["split", column], observed=True).size().rename("n").reset_index()
    totals = df.groupby("split", observed=True).size().rename("split_n")
    return (
        counts.join(totals, on="split")
        .assign(pct=lambda frame: (100 * frame["n"] / frame["split_n"]).round(1))
        .sort_values(["split", "n", column], ascending=[True, False, True])
        .reset_index(drop=True)
    )


gold_kind_distribution = distribution_table(all_records_df, "gold_label_kind")
purist_distribution = distribution_table(all_records_df, "gold_purist_category")
pragmatic_distribution = distribution_table(all_records_df, "gold_pragmatic_category")

display(gold_kind_distribution)
display(purist_distribution)
display(pragmatic_distribution)

,split,gold_label_kind,n,split_n,pct
0,test,frequency,281,450,62.4
1,test,seizure_free,67,450,14.9
2,test,unknown,60,450,13.3
3,test,unresolved_multiple,26,450,5.8
4,test,no_reference,16,450,3.6
5,train,frequency,188,300,62.7
6,train,seizure_free,44,300,14.7
7,train,unknown,40,300,13.3
8,train,unresolved_multiple,17,300,5.7
9,train,no_reference,11,300,3.7


,split,gold_purist_category,n,split_n,pct
0,test,seizure_freq_unknown,102,450,22.7
1,test,seizure_freq_more1week_less1day,98,450,21.8
2,test,currently_no_seizure,67,450,14.9
3,test,seizure_freq_more1mon_less1week,62,450,13.8
4,test,seizure_freq_more1per6mon_less1mon,52,450,11.6
5,test,seizure_freq_1ormore_daily,36,450,8.0
6,test,seizure_freq_1_per_mon,20,450,4.4
7,test,seizure_freq_1_per_week,6,450,1.3
8,test,seizure_freq_1_per_yr,6,450,1.3
9,test,seizure_freq_1_per_6mon,1,450,0.2


,split,gold_pragmatic_category,n,split_n,pct
0,test,seizure_frequent,202,450,44.9
1,test,seizure_freq_unknown,102,450,22.7
2,test,seizure_infrequent,79,450,17.6
3,test,currently_no_seizure,67,450,14.9
4,train,seizure_frequent,141,300,47.0
5,train,seizure_freq_unknown,68,300,22.7
6,train,seizure_infrequent,47,300,15.7
7,train,currently_no_seizure,44,300,14.7
8,validation,seizure_frequent,346,750,46.1
9,validation,seizure_freq_unknown,170,750,22.7


## Scoring Helpers

Use repo scoring functions for Purist and Pragmatic results. The identity check below is only a smoke test that gold labels round-trip through the scorer.

In [4]:
def metrics_frame(y_true, y_pred) -> pd.DataFrame:
    rows = []
    for method in ("purist", "pragmatic"):
        metrics = evaluate_predictions(y_true, y_pred, method=method)
        for averaging, values in metrics.items():
            rows.append({"method": method, "averaging": averaging, **values})
    return pd.DataFrame(rows)


def score_prediction_frame(
    frame: pd.DataFrame,
    prediction_column: str = "prediction_monthly_frequency",
    gold_column: str = "gold_monthly_frequency",
) -> pd.DataFrame:
    scored = frame.dropna(subset=[gold_column, prediction_column]).copy()
    return metrics_frame(scored[gold_column].astype(float), scored[prediction_column].astype(float))


identity_rows = all_records_df.assign(
    prediction_monthly_frequency=lambda frame: frame["gold_monthly_frequency"]
)
score_prediction_frame(identity_rows)

,method,averaging,precision,recall,f1,accuracy
0,purist,micro,1.0,1.0,1.0,1.0
1,purist,macro,1.0,1.0,1.0,1.0
2,purist,weighted,1.0,1.0,1.0,1.0
3,pragmatic,micro,1.0,1.0,1.0,1.0
4,pragmatic,macro,1.0,1.0,1.0,1.0
5,pragmatic,weighted,1.0,1.0,1.0,1.0


## Candidate Artifact Loader

Point `ARTIFACT_PATH` at a candidate JSONL. The loader understands the structured LLM replay shape used by `gan2026_clean_attribution_format50_v0` and keeps parse/schema failures separate from scorable rows.

In [5]:
ARTIFACT_PATH = REPO_ROOT / "experiments/gan2026_clean_attribution_format50_v0_2026-06-01.jsonl"


def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def artifact_records_to_frame(rows: list[dict]) -> pd.DataFrame:
    flattened = []
    for row in rows:
        comparison = row.get("comparison") or {}
        reference = row.get("reference") or {}
        structured = row.get("structured_record") or {}
        selection = structured.get("selection") or {}
        parse_errors = row.get("parse_errors") or []
        normalized_events = row.get("normalized_events") or []
        event_kinds = [event.get("semantic_kind") for event in normalized_events]
        predicted_monthly = comparison.get("predicted_monthly_frequency")
        flattened.append(
            {
                "source_row_index": row.get("source_row_index"),
                "split": row.get("split"),
                "prompt_version": row.get("prompt_version"),
                "gold_label": reference.get("gold_label"),
                "gold_label_kind": reference.get("gold_label_kind"),
                "gold_monthly_frequency": comparison.get(
                    "gold_monthly_frequency", reference.get("gold_monthly_frequency")
                ),
                "prediction_label": selection.get("final_label"),
                "prediction_kind": selection.get("final_kind"),
                "prediction_monthly_frequency": predicted_monthly,
                "purist_correct": comparison.get("purist_correct"),
                "pragmatic_correct": comparison.get("pragmatic_correct"),
                "gold_purist_category": comparison.get("gold_purist_category"),
                "predicted_purist_category": comparison.get("predicted_purist_category"),
                "gold_pragmatic_category": comparison.get("gold_pragmatic_category"),
                "predicted_pragmatic_category": comparison.get("predicted_pragmatic_category"),
                "evidence_valid": row.get("evidence_valid"),
                "parse_error_count": len(parse_errors),
                "parse_errors": "; ".join(parse_errors),
                "event_count": len(normalized_events),
                "event_kinds": ", ".join(str(kind) for kind in event_kinds if kind),
                "cluster_event_count": sum(kind == "cluster" for kind in event_kinds),
                "selected_evidence": selection.get("evidence"),
                "rationale": selection.get("rationale"),
            }
        )
    return pd.DataFrame(flattened)


artifact_rows = load_jsonl(ARTIFACT_PATH)
artifact_df = artifact_records_to_frame(artifact_rows)
artifact_df.head()

,source_row_index,split,prompt_version,gold_label,gold_label_kind,gold_monthly_frequency,prediction_label,prediction_kind,prediction_monthly_frequency,purist_correct,pragmatic_correct,gold_purist_category,predicted_purist_category,gold_pragmatic_category,predicted_pragmatic_category,evidence_valid,parse_error_count,parse_errors,event_count,event_kinds,cluster_event_count,selected_evidence,rationale
0,10,validation,gan2026_llm_structured_event_selector_v0.5,4 per day,frequency,121.666667,4 per day,frequency,121.666667,True,True,seizure_freq_1ormore_daily,seizure_freq_1ormore_daily,seizure_frequent,seizure_frequent,True,1,final_label_repaired: 'up to 4 per day' -> '4 per day',3,"no_reference, frequency, unknown",0,"On the accommodation logs, the observed frequency is noted as ≤ four per day, with variable clustering","The accommodation logs provide a precise current seizure frequency estimate (up to four per day), which is the highest current seizure b..."
1,40,validation,gan2026_llm_structured_event_selector_v0.5,4 per week,frequency,17.380952,4 per week,frequency,17.380952,True,True,seizure_freq_more1week_less1day,seizure_freq_more1week_less1day,seizure_frequent,seizure_frequent,True,1,final_label_repaired: '≤ 4 per week' -> '4 per week',3,"frequency, unknown, unknown",0,overall a frequency of ≤ four seizures per week,"The note provides an overall current seizure frequency of ≤ four seizures per week, which encompasses all seizure types and is the highe..."
2,79,validation,gan2026_llm_structured_event_selector_v0.5,6 to 7 per year,frequency,0.541667,6 to 7 per year,frequency,0.541667,True,True,seizure_freq_more1per6mon_less1mon,seizure_freq_more1per6mon_less1mon,seizure_infrequent,seizure_infrequent,True,1,final_label_repaired: '≤ 6 to 7 per year' -> '6 to 7 per year',4,"frequency, unknown, unknown, unknown",0,Seizure frequency currently reported as ≤ 6 to 7 per year,"The note provides an overall current seizure frequency estimate of ≤ 6 to 7 per year, which encompasses all seizure types and is the cle..."
3,103,validation,gan2026_llm_structured_event_selector_v0.5,2 to 4 per year,frequency,0.250000,2 to 4 per year,frequency,0.250000,True,True,seizure_freq_more1per6mon_less1mon,seizure_freq_more1per6mon_less1mon,seizure_infrequent,seizure_infrequent,True,0,,2,"frequency, frequency",0,"Over the past year, however, the patient and family report that events have become markedly infrequent, such that the current pattern is...","The current seizure frequency is clearly stated as ≤ two or four per year, indicating a markedly reduced event burden compared to the hi..."
4,128,validation,gan2026_llm_structured_event_selector_v0.5,17 per month,frequency,17.236111,17 per month,frequency,17.236111,True,True,seizure_freq_more1week_less1day,seizure_freq_more1week_less1day,seizure_frequent,seizure_frequent,True,0,,2,"frequency, no_reference",0,He reports a current seizure frequency of 17 per month,"The note provides a clear current overall seizure frequency of 17 per month, which is the highest current seizure burden and most clinic..."


In [12]:
artifact_columns = [
    "gold_label",
    "prediction_label",
    "purist_correct",
    "parse_error_count",
    "parse_errors",
    "event_count",
    "event_kinds",
    "cluster_event_count",
    "selected_evidence",
    "rationale",
]
artifact_df[artifact_columns]

,gold_label,prediction_label,purist_correct,parse_error_count,parse_errors,event_count,event_kinds,cluster_event_count,selected_evidence,rationale
0,4 per day,4 per day,True,1,final_label_repaired: 'up to 4 per day' -> '4 per day',3,"no_reference, frequency, unknown",0,"On the accommodation logs, the observed frequency is noted as ≤ four per day, with variable clustering","The accommodation logs provide a precise current seizure frequency estimate (up to four per day), which is the highest current seizure b..."
1,4 per week,4 per week,True,1,final_label_repaired: '≤ 4 per week' -> '4 per week',3,"frequency, unknown, unknown",0,overall a frequency of ≤ four seizures per week,"The note provides an overall current seizure frequency of ≤ four seizures per week, which encompasses all seizure types and is the highe..."
2,6 to 7 per year,6 to 7 per year,True,1,final_label_repaired: '≤ 6 to 7 per year' -> '6 to 7 per year',4,"frequency, unknown, unknown, unknown",0,Seizure frequency currently reported as ≤ 6 to 7 per year,"The note provides an overall current seizure frequency estimate of ≤ 6 to 7 per year, which encompasses all seizure types and is the cle..."
3,2 to 4 per year,2 to 4 per year,True,0,,2,"frequency, frequency",0,"Over the past year, however, the patient and family report that events have become markedly infrequent, such that the current pattern is...","The current seizure frequency is clearly stated as ≤ two or four per year, indicating a markedly reduced event burden compared to the hi..."
4,17 per month,17 per month,True,0,,2,"frequency, no_reference",0,He reports a current seizure frequency of 17 per month,"The note provides a clear current overall seizure frequency of 17 per month, which is the highest current seizure burden and most clinic..."
5,1 per 6 day,1 per 6 day,True,1,final_label_repaired: '1 per 6 days' -> '1 per 6 day',1,frequency,0,"Patient reports seizures every 6 days, typically brief focal aware episodes with auditory distortion and right-sided facial tingling.","The note provides a clear current seizure frequency of one seizure every 6 days, corroborated by family over the past two months, repres..."
6,1 per 7 day,1 per week,True,0,,2,"frequency, unknown",0,a pattern of seizures every seven days,"The note explicitly states a current seizure frequency of every seven days, which is the clearest and most specific frequency informatio..."
7,1 per 2 day,1 per 2 day,True,1,final_label_repaired: '1 per 2 days' -> '1 per 2 day',2,"frequency, no_reference",0,seizures are occurring every 2 days on average,"The note clearly states a current seizure frequency of every 2 days based on consistent seizure logs, representing the highest current s..."
8,1 per 7 to 9 day,1 cluster per week,None,1,unscorable_final_label: Unparsable cluster label: '1 cluster per week',4,"frequency, unknown, no_reference, unknown",0,events tend to cluster every seven to nine days,"The note indicates seizure events cluster every 7-9 days, which is the highest current seizure burden described. The overall cluster fre..."
9,1 per 4 week,1 cluster per 4 week,None,2,final_label_repaired: '1 cluster per 4 weeks' -> '1 cluster per 4 week'; unscorable_final_label: Unparsable cluster label: '1 cluster pe...,3,"frequency, unknown, no_reference",0,"he reports clusters of brief absence episodes every 4 weeks, usually over 1–2 days",Current seizure burden is best represented by the ongoing absence clusters every 4 weeks; GTCS are historical and currently absent.


In [6]:
if artifact_df.empty:
    print(f"No artifact rows found at {ARTIFACT_PATH}")
else:
    artifact_summary = pd.DataFrame(
        [
            {
                "rows": len(artifact_df),
                "scorable_rows": int(artifact_df["prediction_monthly_frequency"].notna().sum()),
                "purist_correct_rows": int(artifact_df["purist_correct"].eq(True).sum()),
                "pragmatic_correct_rows": int(artifact_df["pragmatic_correct"].eq(True).sum()),
                "purist_all_row_accuracy": round(
                    artifact_df["purist_correct"].eq(True).mean(), 4
                ),
                "pragmatic_all_row_accuracy": round(
                    artifact_df["pragmatic_correct"].eq(True).mean(), 4
                ),
                "evidence_valid_rows": int(artifact_df["evidence_valid"].eq(True).sum()),
                "parse_error_rows": int((artifact_df["parse_error_count"] > 0).sum()),
            }
        ]
    )
    display(artifact_summary)
    display(score_prediction_frame(artifact_df))

,rows,scorable_rows,purist_correct_rows,pragmatic_correct_rows,purist_all_row_accuracy,pragmatic_all_row_accuracy,evidence_valid_rows,parse_error_rows
0,50,47,41,43,0.82,0.86,50,19


,method,averaging,precision,recall,f1,accuracy
0,purist,micro,0.8723,0.8723,0.8723,0.8723
1,purist,macro,0.9101,0.9148,0.9043,0.8723
2,purist,weighted,0.8989,0.8723,0.8756,0.8723
3,pragmatic,micro,0.9149,0.9149,0.9149,0.9149
4,pragmatic,macro,0.9399,0.9210,0.9286,0.9149
5,pragmatic,weighted,0.9160,0.9149,0.9134,0.9149


## Failure Slices

Start with scorer-visible failures, then sort by clinical/debugging families. Keep examples row-level enough to explain behavior without turning the status file into a remediation log.

In [7]:
def failure_slices(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    if frame.empty:
        return {}
    failures = frame[frame["purist_correct"] != True].copy()  # noqa: E712
    return {
        "by_gold_kind": (
            failures.groupby("gold_label_kind", dropna=False)
            .size()
            .rename("n")
            .reset_index()
        ),
        "by_prediction_kind": (
            failures.groupby("prediction_kind", dropna=False)
            .size()
            .rename("n")
            .reset_index()
        ),
        "by_confusion": (
            failures.groupby(["gold_purist_category", "predicted_purist_category"], dropna=False)
            .size()
            .rename("n")
            .reset_index()
            .sort_values("n", ascending=False)
        ),
        "parse_or_schema": frame[frame["parse_error_count"] > 0][
            ["source_row_index", "prediction_label", "gold_label", "parse_errors"]
        ],
        "evidence_invalid": frame[~frame["evidence_valid"]][
            ["source_row_index", "prediction_label", "gold_label", "selected_evidence"]
        ],
    }


slices = failure_slices(artifact_df)
for name, table in slices.items():
    print(f"\n{name}")
    display(table)


by_gold_kind


,gold_label_kind,n
0,frequency,8
1,unresolved_multiple,1



by_prediction_kind


,prediction_kind,n
0,frequency,9



by_confusion


,gold_purist_category,predicted_purist_category,n
2,seizure_freq_more1per6mon_less1mon,seizure_freq_more1mon_less1week,3
4,NaN,NaN,3
0,seizure_freq_more1mon_less1week,seizure_freq_1_per_mon,1
1,seizure_freq_more1mon_less1week,seizure_freq_more1week_less1day,1
3,seizure_freq_more1week_less1day,seizure_freq_more1mon_less1week,1



parse_or_schema


,source_row_index,prediction_label,gold_label,parse_errors
0,10,4 per day,4 per day,final_label_repaired: 'up to 4 per day' -> '4 per day'
1,40,4 per week,4 per week,final_label_repaired: '≤ 4 per week' -> '4 per week'
2,79,6 to 7 per year,6 to 7 per year,final_label_repaired: '≤ 6 to 7 per year' -> '6 to 7 per year'
5,156,1 per 6 day,1 per 6 day,final_label_repaired: '1 per 6 days' -> '1 per 6 day'
7,182,1 per 2 day,1 per 2 day,final_label_repaired: '1 per 2 days' -> '1 per 2 day'
8,187,1 cluster per week,1 per 7 to 9 day,unscorable_final_label: Unparsable cluster label: '1 cluster per week'
9,190,1 cluster per 4 week,1 per 4 week,final_label_repaired: '1 cluster per 4 weeks' -> '1 cluster per 4 week'; unscorable_final_label: Unparsable cluster label: '1 cluster pe...
12,218,1 per 3 week,1 per 3 week,final_label_repaired: '1 per 3 weeks' -> '1 per 3 week'
13,243,1 per 4 month,1 per 4 month,final_label_repaired: '1 per 4 months' -> '1 per 4 month'
17,409,1 per month,1 per month,final_label_repaired: '1 per month or less' -> '1 per month'



evidence_invalid


,source_row_index,prediction_label,gold_label,selected_evidence


## Row Review Queue

Use this compact queue for targeted row reading. Add columns as new artifact families expose richer diagnostics, but keep source text review disciplined by split policy.

In [8]:
if artifact_df.empty:
    review_queue = pd.DataFrame()
else:
    review_queue = (
        artifact_df[artifact_df["purist_correct"] != True]  # noqa: E712
        [[
            "source_row_index",
            "gold_label",
            "prediction_label",
            "gold_purist_category",
            "predicted_purist_category",
            "parse_errors",
            "selected_evidence",
            "rationale",
        ]]
        .sort_values(["parse_errors", "source_row_index"], ascending=[False, True])
        .reset_index(drop=True)
    )

review_queue

,source_row_index,gold_label,prediction_label,gold_purist_category,predicted_purist_category,parse_errors,selected_evidence,rationale
0,744,multiple per week,most weekdays,NaN,NaN,unscorable_final_label: Unparsable label (raw: 'most weekdays' / normalized: 'most weekdays'),"brief absences occurring on most weekdays, often clustering around late afternoon","The patient currently experiences frequent typical absence seizures on most weekdays, which represents the highest current seizure burde..."
1,187,1 per 7 to 9 day,1 cluster per week,NaN,NaN,unscorable_final_label: Unparsable cluster label: '1 cluster per week',events tend to cluster every seven to nine days,"The note indicates seizure events cluster every 7-9 days, which is the highest current seizure burden described. The overall cluster fre..."
2,190,1 per 4 week,1 cluster per 4 week,NaN,NaN,final_label_repaired: '1 cluster per 4 weeks' -> '1 cluster per 4 week'; unscorable_final_label: Unparsable cluster label: '1 cluster pe...,"he reports clusters of brief absence episodes every 4 weeks, usually over 1–2 days",Current seizure burden is best represented by the ongoing absence clusters every 4 weeks; GTCS are historical and currently absent.
3,212,1 per 3 to 4 week,1 per month,seizure_freq_more1mon_less1week,seizure_freq_1_per_mon,,ongoing episodes occurring every 3 - 4 weeks,"The note provides a clear current seizure frequency of ongoing episodes every 3-4 weeks, which is the highest current seizure burden des..."
4,665,2 per 2 week,2 per month,seizure_freq_more1week_less1day,seizure_freq_more1mon_less1week,,The app logs indicate a regular pattern of seizures twice every two weeks,"The note provides a clear, current, and quantified seizure frequency from the patient's seizure diary app over the past four months, whi..."
5,790,1 per 7 to 10 day,1 per week,seizure_freq_more1mon_less1week,seizure_freq_more1week_less1day,,"events have continued at a fairly regular cadence, occurring roughly once every seven to ten days","The overall seizure frequency is best represented by the patient's report of events occurring roughly once every seven to ten days, whic..."
6,959,1 per 2 month,2 per month,seizure_freq_more1per6mon_less1mon,seizure_freq_more1mon_less1week,,"She notes the events are occurring bimonthly on average, though some months she has none and then two in quick succession.","The note provides a clear current seizure frequency estimate as 'bimonthly on average' with corroborated diary data, indicating approxim..."
7,960,1 per 2 month,2 to 3 per month,seizure_freq_more1per6mon_less1mon,seizure_freq_more1mon_less1week,,ongoing events occurring with bimonthly seizures,"The note clearly states ongoing bimonthly seizures, indicating approximately 2 to 3 seizures per month, which is the highest current sei..."
8,987,1 per 2 month,2 per month,seizure_freq_more1per6mon_less1mon,seizure_freq_more1mon_less1week,,bimonthly seizures,"The note explicitly states 'bimonthly seizures' as the current overall seizure frequency despite adherence to medication, which is the c..."
